# SOLUTION (R version): Chi-Square Test – Expanded Workflow


## Flowchart: Choosing the Right Test for Categorical Associations
```mermaid
flowchart TD
    A[Two Categorical Variables?] --> B{Create Contingency Table}
    B --> C{Expected counts < 5 in many cells?}
    C -->|Yes| D[Use Fisher's Exact Test]
    C -->|No| E[Run chisq.test()]
    E --> F{p.value < 0.05?}
    F -->|Yes| G[Calculate Cramér's V]
    F -->|No| H[No significant association]
    G --> I[Post-hoc pairwise tests with correction]
    I --> J[Report with audience in mind]
    H --> J
```
**Practical tip:** Always report p-value + effect size (Cramér's V).


## 1. Load Data and Table (Solution)


In [ ]:
library(tidyverse)
library(lsr)

ants <- read_csv("ants_grade.csv")
table <- table(ants$Grade, ants$Ant)
print(table)
print(addmargins(table))


## 2. Chi-Square Test (Solution)

**Result:** p-value ≈ 0.084 → Not significant at α = 0.05.


In [ ]:
chi_result <- chisq.test(table)
print(chi_result)
print(round(chi_result$expected, 2))

significant <- chi_result$p.value < 0.05
cat("\nSignificant association?", significant, "\n")


## 3. Cramér's V (Solution)

**Cramér's V ≈ 0.20** → Small to medium effect.


In [ ]:
cramers_v <- cramersV(table)
cat("Cramér's V =", round(cramers_v, 3), "(small-medium effect)\n")


## 4. Assumptions & Post-hoc (Solution)

All expected frequencies > 5 → assumptions met.
Since overall test is not significant, we do not perform post-hoc tests.


In [ ]:
cat("Minimum expected frequency:", min(chi_result$expected), "\n")
cat("Any expected < 5?", any(chi_result$expected < 5), "\n")

cat("\nOverall test not significant → no post-hoc pairwise tests needed.\n")


## 5. More Practice (Solution)

**Goodness of Fit on Ant type:**


In [ ]:
ant_counts <- table(ants$Ant)
gof <- chisq.test(ant_counts)
print(gof)
cat("Harvesters are significantly more popular overall.\n")


## 6. Simulation (Solution)


In [ ]:
set.seed(42)

n_per_grade <- 36
prob_leaf_null <- 0.24
prob_leaf_3rd <- 0.36
n_simulations <- 300

sig_count <- 0

for (i in 1:n_simulations) {
  ants_sim <- c()
  for (grade in c("1st", "2nd", "3rd")) {
    p_leaf <- ifelse(grade == "3rd", prob_leaf_3rd, prob_leaf_null)
    n_leaf <- rbinom(1, n_per_grade, p_leaf)
    ants_sim <- c(ants_sim, rep("leaf cutter", n_leaf), rep("harvester", n_per_grade - n_leaf))
  }
  
  df_sim <- data.frame(
    Grade = rep(c("1st", "2nd", "3rd"), each = n_per_grade),
    Ant = ants_sim
  )
  
  tab <- table(df_sim$Grade, df_sim$Ant)
  p <- chisq.test(tab)$p.value
  if (p < 0.05) sig_count <- sig_count + 1
}

cat("Power to detect association:", round(sig_count / n_simulations, 3), "\n")


## 7. Practical Conclusion (Solution)

### For School Administrators
> "We did not find strong statistical evidence that different grade levels prefer different ant species (p = 0.084). However, there was a small-to-medium association (Cramér's V = 0.20). Third graders showed relatively more interest in Leaf Cutters. We recommend continuing to offer both species but perhaps promoting Leaf Cutters more to older students."

### Technical Version
Chi-Square test: χ²(2) = 4.97, p = 0.084, Cramér's V = 0.20. All expected frequencies satisfactory. No significant association detected, though a modest effect size suggests possible differences in 3rd grade preferences.
